# RSI Mean Reversion

Tests a 14-day Relative Strength Index entry and exit rule on AAPL using adjusted closing prices. Compares the long-or-cash strategy with buy and hold before and after January 2020. The backtest excludes trading costs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from _market_data import download_ohlcv

def calculate_rsi(prices, period=14):
    """Calculate RSI (Relative Strength Index)"""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def backtest_rsi_strategy(data, rsi_oversold=30, rsi_overbought=70):
    """
    Test an RSI entry and exit rule:
    - Generate signals based on previous day's RSI
    - Buy when RSI < oversold (30)
    - Sell when RSI > overbought (70)
    """
    df = data.copy()
    df['position'] = 0.0
    df['signal'] = 0
    df['trade'] = 0
    
    # Generate signals from closing prices
    df.loc[df['RSI'] < rsi_oversold, 'signal'] = 1  # Buy signal
    df.loc[df['RSI'] > rsi_overbought, 'signal'] = -1  # Sell signal
    
    # Update positions on the following session
    position = 0.0
    positions = np.zeros(len(df))
    
    for i in range(1, len(df)):
        prev_signal = df['signal'].iloc[i-1]  # Use previous day's signal
        
        if prev_signal == 1 and position == 0:  # Buy
            position = 1.0
            df.loc[df.index[i], 'trade'] = 1
        elif prev_signal == -1 and position == 1:  # Sell
            position = 0.0
            df.loc[df.index[i], 'trade'] = -1
            
        positions[i] = position
    
    df['position'] = positions
    
    # Calculate returns
    df['returns'] = df['Close'].pct_change()
    df['strategy_returns'] = df['position'].shift(1) * df['returns']
    
    # Fix NaN values
    df['strategy_returns'] = df['strategy_returns'].fillna(0)
    df['returns'] = df['returns'].fillna(0)
    
    df['cumulative_returns'] = (1 + df['returns']).cumprod()
    df['cumulative_strategy'] = (1 + df['strategy_returns']).cumprod()
    
    return df

def calculate_metrics(returns):
    """Calculate performance metrics"""
    returns_clean = returns.dropna()
    if len(returns_clean) == 0:
        return {}
        
    total_return = (1 + returns_clean).prod() - 1
    annualized_return = (1 + total_return) ** (252 / len(returns_clean)) - 1
    volatility = returns_clean.std() * np.sqrt(252)
    sharpe_ratio = annualized_return / volatility if volatility != 0 else 0
    
    # Maximum drawdown
    cumulative = (1 + returns_clean).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Win rate
    positive_returns = returns_clean[returns_clean > 0]
    win_rate = len(positive_returns) / len(returns_clean) if len(returns_clean) > 0 else 0
    
    return {
        'Total Return': f"{total_return:.2%}",
        'Annualized Return': f"{annualized_return:.2%}",
        'Volatility': f"{volatility:.2%}",
        'Sharpe Ratio': f"{sharpe_ratio:.3f}",
        'Max Drawdown': f"{max_drawdown:.2%}",
        'Win Rate': f"{win_rate:.2%}"
    }

# Download AAPL data
ticker = "AAPL"
print(f"Downloading {ticker} data...")
start_date = "2016-12-01"  # Start earlier to have enough data for RSI calculation
end_date = "2022-12-31"

data = download_ohlcv(ticker, start_date, end_date, auto_adjust=True, end_inclusive=False)

# Calculate RSI
data['RSI'] = calculate_rsi(data['Close'])

# Remove initial rows with NaN RSI values
data = data.dropna()

# Split into periods (ensure clean split)
split_date = '2020-01-01'
in_sample = data[data.index < split_date].copy()
out_sample = data[data.index >= split_date].copy()
if in_sample.empty or out_sample.empty:
    raise ValueError("Choose dates that provide data before and after the split date.")

print(f"\nIn-Sample Period: {in_sample.index[0].date()} to {in_sample.index[-1].date()}")
print(f"Out-of-Sample Period: {out_sample.index[0].date()} to {out_sample.index[-1].date()}")

# In-Sample Backtest (2017-2019)
print("\n" + "="*50)
print("IN-SAMPLE BACKTEST")
print("="*50)

in_sample_results = backtest_rsi_strategy(in_sample)

# In-Sample Metrics
buy_hold_returns_in = in_sample_results['returns'].dropna()
strategy_returns_in = in_sample_results['strategy_returns'].dropna()

print("\nBuy & Hold Metrics (In-Sample):")
buy_hold_metrics_in = calculate_metrics(buy_hold_returns_in)
for key, value in buy_hold_metrics_in.items():
    print(f"{key}: {value}")

print("\nRSI Strategy Metrics (In-Sample):")
strategy_metrics_in = calculate_metrics(strategy_returns_in)
for key, value in strategy_metrics_in.items():
    print(f"{key}: {value}")

# Out-of-Sample Backtest (2020-2022)
print("\n" + "="*50)
print("OUT-OF-SAMPLE BACKTEST")
print("="*50)

out_sample_results = backtest_rsi_strategy(out_sample)

# Out-of-Sample Metrics
buy_hold_returns_out = out_sample_results['returns'].dropna()
strategy_returns_out = out_sample_results['strategy_returns'].dropna()

print("\nBuy & Hold Metrics (Out-of-Sample):")
buy_hold_metrics_out = calculate_metrics(buy_hold_returns_out)
for key, value in buy_hold_metrics_out.items():
    print(f"{key}: {value}")

print("\nRSI Strategy Metrics (Out-of-Sample):")
strategy_metrics_out = calculate_metrics(strategy_returns_out)
for key, value in strategy_metrics_out.items():
    print(f"{key}: {value}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(f'{ticker} Relative Strength Index Strategy', fontsize=16)

# In-Sample: Price and RSI
ax1 = axes[0, 0]
ax1.plot(in_sample_results.index, in_sample_results['Close'], label=f'{ticker} Price', color='blue')
ax1.set_title(f'In-Sample: {ticker} Price')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left')

# Add buy/sell markers
buy_signals = in_sample_results[in_sample_results['trade'] == 1]
sell_signals = in_sample_results[in_sample_results['trade'] == -1]
ax1.scatter(buy_signals.index, buy_signals['Close'], color='green', marker='^', s=50, label='Buy')
ax1.scatter(sell_signals.index, sell_signals['Close'], color='red', marker='v', s=50, label='Sell')

ax1_rsi = ax1.twinx()
ax1_rsi.plot(in_sample_results.index, in_sample_results['RSI'], label='RSI', color='orange', alpha=0.7)
ax1_rsi.axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Overbought')
ax1_rsi.axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Oversold')
ax1_rsi.set_ylabel('RSI')
ax1_rsi.set_ylim(0, 100)
ax1_rsi.legend(loc='upper right')

# In-Sample: Cumulative Returns
axes[0, 1].plot(in_sample_results.index, in_sample_results['cumulative_returns'], 
                label='Buy & Hold', color='blue', linewidth=2)
axes[0, 1].plot(in_sample_results.index, in_sample_results['cumulative_strategy'], 
                label='RSI Strategy', color='red', linewidth=2)
axes[0, 1].set_title('In-Sample: Cumulative Returns')
axes[0, 1].set_ylabel('Cumulative Returns')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Out-of-Sample: Price and RSI
ax3 = axes[1, 0]
ax3.plot(out_sample_results.index, out_sample_results['Close'], label=f'{ticker} Price', color='blue')
ax3.set_title(f'Out-of-Sample: {ticker} Price')
ax3.set_ylabel('Price ($)')
ax3.legend(loc='upper left')

# Add buy/sell markers
buy_signals_out = out_sample_results[out_sample_results['trade'] == 1]
sell_signals_out = out_sample_results[out_sample_results['trade'] == -1]
ax3.scatter(buy_signals_out.index, buy_signals_out['Close'], color='green', marker='^', s=50, label='Buy')
ax3.scatter(sell_signals_out.index, sell_signals_out['Close'], color='red', marker='v', s=50, label='Sell')

ax3_rsi = ax3.twinx()
ax3_rsi.plot(out_sample_results.index, out_sample_results['RSI'], label='RSI', color='orange', alpha=0.7)
ax3_rsi.axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Overbought')
ax3_rsi.axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Oversold')
ax3_rsi.set_ylabel('RSI')
ax3_rsi.set_ylim(0, 100)
ax3_rsi.legend(loc='upper right')

# Out-of-Sample: Cumulative Returns
axes[1, 1].plot(out_sample_results.index, out_sample_results['cumulative_returns'], 
                label='Buy & Hold', color='blue', linewidth=2)
axes[1, 1].plot(out_sample_results.index, out_sample_results['cumulative_strategy'], 
                label='RSI Strategy', color='red', linewidth=2)
axes[1, 1].set_title('Out-of-Sample: Cumulative Returns')
axes[1, 1].set_ylabel('Cumulative Returns')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Trade Analysis
print("\n" + "="*60)
print("TRADE ANALYSIS")
print("="*60)

# In-Sample trades
buy_trades_in = (in_sample_results['trade'] == 1).sum()
sell_trades_in = (in_sample_results['trade'] == -1).sum()

print(f"\nIn-Sample Trades:")
print(f"Buy signals: {buy_trades_in}")
print(f"Sell signals: {sell_trades_in}")
print(f"Complete round trips: {min(buy_trades_in, sell_trades_in)}")

# Out-of-Sample trades
buy_trades_out = (out_sample_results['trade'] == 1).sum()
sell_trades_out = (out_sample_results['trade'] == -1).sum()

print(f"\nOut-of-Sample Trades:")
print(f"Buy signals: {buy_trades_out}")
print(f"Sell signals: {sell_trades_out}")
print(f"Complete round trips: {min(buy_trades_out, sell_trades_out)}")

# Final Summary
print("\n" + "="*60)
print("SUMMARY COMPARISON")
print("="*60)
print("\nIN-SAMPLE:")
print(f"Buy & Hold Return: {buy_hold_metrics_in.get('Total Return', 'N/A')}")
print(f"RSI Strategy Return: {strategy_metrics_in.get('Total Return', 'N/A')}")
print(f"RSI Sharpe Ratio: {strategy_metrics_in.get('Sharpe Ratio', 'N/A')}")

print("\nOUT-OF-SAMPLE:")
print(f"Buy & Hold Return: {buy_hold_metrics_out.get('Total Return', 'N/A')}")
print(f"RSI Strategy Return: {strategy_metrics_out.get('Total Return', 'N/A')}")
print(f"RSI Sharpe Ratio: {strategy_metrics_out.get('Sharpe Ratio', 'N/A')}")

print(f"\nStrategy Consistency: ", end="")
try:
    in_ret = float(strategy_metrics_in.get('Total Return', '0%').replace('%', ''))
    out_ret = float(strategy_metrics_out.get('Total Return', '0%').replace('%', ''))
    if in_ret > 0 and out_ret > 0:
        print("Both periods profitable")
    elif in_ret > 0 and out_ret <= 0:
        print("In-sample profitable, out-of-sample unprofitable")
    elif in_ret <= 0 and out_ret > 0:
        print("In-sample unprofitable, out-of-sample profitable")
    else:
        print("Both periods unprofitable")
except ValueError:
    print("Unable to assess consistency")